# Tree-of-Shapes Contour Extraction

This notebook documents how contours are extracted from a weighted tree of shapes. The examples focus on how contours are associated with nodes and how special cases are handled when a contour touches a non-comparable region.


In [ ]:
import mmcfilters
from collections import defaultdict
import math

import cv2 as cv
from IPython import display
import matplotlib
import matplotlib.pyplot as plt
import mtviz as viz
import numpy as np

plt.rcParams["figure.figsize"] = (16, 16)


def make_print_tree(tree):
    return viz.PrintTree(lambda n: tree.getChildren(n), lambda n: f"id:{n}, altitude: {tree.getAltitude(n)}")


def dfs(tree, node, depth=0):
    nodes_by_depth[depth].append(node)
    for child in tree.getChildren(node):
        dfs(tree, child, depth + 1)


def map_ids(tos):
    owner_ids = []
    for pixel in range(num_rows * num_cols):
        owner_ids.append(tos.getProperPartOwner(pixel))
    return np.array(owner_ids).reshape(num_rows, num_cols)


def plot_contours(tree, pixels, node, num_rows, num_cols):
    contour_image = tree.reconstructNode(node)
    for pixel in tree.getConnectedComponent(node):
        row = pixel // num_cols
        col = pixel % num_cols
        contour_image[row, col] = 128
    for pixel in pixels:
        row = pixel // num_cols
        col = pixel % num_cols
        contour_image[row, col] = 255
        if row == 8 and col == 8:
            contour_image[row, col] = 200
    plt.figure(figsize=(5, 5))
    plt.imshow(contour_image, cmap="Reds", vmax=255, vmin=0)
    plt.title(node)
    plt.axis("off")
    plt.show()


def plot_binary_images(tree, contours, nodes_by_depth, num_rows, num_cols):
    num_images = len(nodes_by_depth)
    fig, axes = plt.subplots(num_images, 2, figsize=(5, 10))
    for i, depth in enumerate(range(num_images - 1, -1, -1)):
        binary_image = np.zeros((num_rows, num_cols), dtype=np.uint8)
        contour_image = np.zeros((num_rows, num_cols), dtype=np.uint8)
        for node in nodes_by_depth[depth]:
            for pixel in contours.getContour(node):
                row = pixel // num_cols
                col = pixel % num_cols
                contour_image[row, col] = 1
            for pixel in tree.getConnectedComponent(node):
                row = pixel // num_cols
                col = pixel % num_cols
                binary_image[row, col] = 1
        axes[i, 0].imshow(binary_image, cmap="gray_r", vmax=1, vmin=0, interpolation="nearest")
        axes[i, 1].imshow(contour_image, cmap="Reds")
        axes[i, 0].set_title(f"Depth: {depth}")
        axes[i, 0].axis("off")
    plt.show()


## Build the weighted tree of shapes

The input image is converted into a tree of shapes with a fixed adjacency radius. The contour data structure maps each tree node to the boundary pixels of its corresponding shape.


In [ ]:
input_image = np.array([
    [10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10],
    [10, 10,215,255,255,150,150, 10, 10, 10, 10, 10, 10, 10, 10],
    [10,150,255,255,255,255,255,150, 10, 10, 10, 10, 10, 10, 10],
    [10,215,225,150,225,255,255,255,255,150,150,150, 10, 10, 10],
    [10,225,225,150,150,215,255,255,255,255,255,255,150, 10, 10],
    [10,225,255,150, 10, 10,150,255,255,255,255,255,150, 10, 10],
    [10,215,255,150, 10, 10, 10,150,225,255,255,255,255,150,10],
    [10,150,255,255, 10, 10, 10,150,225,255,255,255,255,215,10],
    [10,150,255,255,150, 10, 10,150,225,150,150,150,255,150,10],
    [10,150,255,255,150, 10, 10,215,215, 10, 10, 10,150, 10,10],
    [10,150,255,255,150, 10,150,215,150, 10, 10, 10, 10, 10,10],
    [10, 10,150,255,225,150,225,215,150, 10, 10, 10, 10, 10,10],
    [10, 10,150,255,255,255,225,150, 10, 10, 10, 10, 10, 10,10],
    [10, 10,150,255,255,255,150, 10, 10, 10, 10, 10, 10, 10,10],
    [10, 10, 10,150,225,255,150, 10, 10, 10, 10, 10, 10, 10,10],
    [10, 10, 10, 10,150,150, 10, 10, 10, 10, 10, 10, 10, 10,10],
    [10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,10]
], dtype=np.uint8)
input_image = np.ascontiguousarray(input_image, dtype=np.uint8)
(num_rows, num_cols) = input_image.shape
#input_image = 255 - input_image

tos = mmcfilters.MorphologicalTreeFactory.createTreeOfShapes(input_image, interpolation=mmcfilters.ToSInterpolation.SelfDual)
print("#Nodes:", tos.numNodes)
nodes_by_depth = defaultdict(list)
dfs(tos, tos.getRoot())
print(input_image)
make_print_tree(tos)(tos.getRoot())
print("\n", map_ids(tos))



tree_contours = mmcfilters.ContourComputation.extraction(tos)

## Plot contours for each node

Iterating over `contoursByNode` makes it possible to inspect every node contour independently. This is useful for debugging the relationship between tree topology and boundary extraction.


In [ ]:

for (nodeId, contour) in tree_contours.contoursByNode():
    plot_contours(tos, contour, nodeId, num_rows, num_cols)

## Case study: contour propagation


### Case 1: contour during processing

The first case highlights a node while the contour extraction logic is still deciding which boundary pixels belong to it.


In [ ]:
# 1. During processing
node = 3
plot_contours(tos, tree_contours.getContour(node), node, num_rows, num_cols)

### Case 2: non-comparable neighbor

This case shows a pixel relationship where neighboring regions do not have a direct ancestor-descendant relationship in the tree.


In [ ]:
# 2. Found a non-comparable neighbor
node = 6
plot_contours(tos, tree_contours.getContour(node), node, num_rows, num_cols)

### Case 3: least common ancestor

When two regions are non-comparable, their least common ancestor identifies the tree node where the ambiguity can be resolved.


In [ ]:
#3. LCA
node = 2
plot_contours(tos, tree_contours.getContour(node), node, num_rows, num_cols)

### Case 4: ancestor contour correction

The last case removes a pixel from the contour of an ancestor when that pixel belongs to a more specific node contour.


In [ ]:
# 4. The pixel to remove belongs to the contour of an LCA ancestor, so it is propagated to that ancestor
node = 1
plot_contours(tos, tree_contours.getContour(node), node, num_rows, num_cols)